# NumPy: arreglos y operaciones vectorizadas

Representaremos rondas de temperatura de tres salas. Primero aprenderemos a leer
la forma de un arreglo; después aplicaremos operaciones a todas las mediciones y
reutilizaremos los resultados en una aplicación.

## Contenido

1. Listas y arreglos; dimensiones y tipos.
2. Selección y máscaras.
3. Vistas y copias.
4. Operaciones vectorizadas, resúmenes por eje y broadcasting.
5. Mediciones incompletas y paso a una aplicación.

Abre esta notebook en VS Code desde `01-exploracion` y selecciona el kernel de `.venv`.
Los ejercicios tienen celdas independientes: puedes recorrer los ejemplos sin
completarlos y resolver cada ejercicio en su propia celda.


In [1]:
import sys
from pathlib import Path

import numpy as np

print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"NumPy: {np.__version__}")

data_dir = Path.cwd().parent / "datos"
if not (data_dir / "readings.csv").is_file():
    raise FileNotFoundError(
        "Start the notebook from the 01-exploracion project directory"
    )

Python: C:\Users\MOISES INFORMATICA\.codex\worktrees\a92c\ia-programacion-uady-main\unidades\02-datos-y-representaciones\tareas\sesion-01-numpy\01-exploracion\.venv\Scripts\python.exe
Working directory: C:\Users\MOISES INFORMATICA\.codex\worktrees\a92c\ia-programacion-uady-main\unidades\02-datos-y-representaciones\tareas\sesion-01-numpy\01-exploracion
NumPy: 2.5.3


## 1. Listas y arreglos

NumPy ofrece `ndarray`, un arreglo de elementos con un tipo común. Su utilidad
no se limita a escribir menos ciclos: también hace explícita la organización de
los datos. Una lista sigue siendo útil cuando necesitamos una colección flexible.

Predice qué devuelve cada multiplicación: una opera sobre una lista y la otra
sobre sus valores numéricos.

[Introducción oficial a NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html).

In [2]:
temperature_list = [22.0, 24.0, 20.0]
temperature_array = np.array(temperature_list)
print(temperature_list * 2)
print(temperature_array * 2)

[22.0, 24.0, 20.0, 22.0, 24.0, 20.0]
[44. 48. 40.]


## 2. Dimensiones, forma y tamaño

Usaremos este orden de columnas: sala norte, sala sur y almacén. Cada fila
corresponde a una ronda en la que se midieron las tres salas.

| Ronda | `north_room` | `south_room` | `storage_room` |
|---|---:|---:|---:|
| 0 | 22 | 24 | 20 |
| 1 | 23 | 25 | 21 |
| 2 | 24 | 26 | 22 |

`loadtxt` lee valores numéricos; `skiprows=1` omite el encabezado y `ndmin=2`
mantiene al menos dos dimensiones, incluso para un archivo de una sola fila.
El orden de los nombres se conserva por acuerdo con el archivo, no dentro del
arreglo. La [ficha de datos](../datos/README.md) describe las unidades y su origen.

[Referencia de loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [3]:
room_names = ("north_room", "south_room", "storage_room")
readings = np.loadtxt(data_dir / "readings.csv", delimiter=",", skiprows=1, ndmin=2)
print(readings)

[[22. 24. 20.]
 [23. 25. 21.]
 [24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]
 [27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]


`ndim` cuenta ejes; `shape` indica cuántas posiciones hay en cada eje; `size`
es el total de elementos. `nbytes` cuenta los bytes del búfer de datos, no todos
los objetos ni el consumo total de Python.

```text
                        axis 1: salas →
                      norte   sur   almacén
axis 0: rondas ↓    0   22     24      20
                   1   23     25      21
                   2   24     26      22
                   …
```

[Propiedades de ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html).

In [4]:
print(f"Dimensions: {readings.ndim}")
print(f"Shape: {readings.shape}")
print(f"Elements: {readings.size}")
print(f"Data type: {readings.dtype}")
print(f"Bytes per element: {readings.itemsize}")
print(f"Data bytes: {readings.nbytes}")
assert readings.size == 8 * 3

Dimensions: 2
Shape: (8, 3)
Elements: 24
Data type: float64
Bytes per element: 8
Data bytes: 192


In [5]:
scalar = np.array(22.0)
vector = np.array([22.0, 24.0, 20.0])
matrix = np.array([[22.0, 24.0, 20.0]])
for name, values in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(name, values.ndim, values.shape, values.size)

scalar 0 () 1
vector 1 (3,) 3
matrix 2 (1, 3) 3


### Ejercicio 1: una ronda, dos representaciones

Construye `one_round` como vector de tres temperaturas y `one_round_table`
como matriz de una fila. Comprueba las formas `(3,)` y `(1, 3)`. Explica por qué
ambos tienen tres elementos y distinto número de ejes.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [6]:
one_round = np.array([22.0, 24.0, 20.0])
one_round_table = np.array([[22.0, 24.0, 20.0]])

assert one_round.shape == (3,)
assert one_round_table.shape == (1, 3)
assert one_round.size == one_round_table.size == 3
assert one_round.ndim == 1
assert one_round_table.ndim == 2
print("Vector:", one_round.shape, "Table:", one_round_table.shape)
# Los dos arreglos contienen tres valores, pero la tabla conserva un eje de filas.


Vector: (3,) Table: (1, 3)


## 3. Tipos numéricos

El tipo del arreglo determina cómo se representa cada elemento. `int64` guarda
enteros; `float64` permite decimales y valores especiales como `nan`.
Mezclar números con texto puede producir un arreglo de cadenas: crear el arreglo
no garantiza que pueda usarse en operaciones numéricas.

La asignación siguiente no convierte automáticamente el arreglo entero en uno
flotante. Observa qué sucede con la fracción. Para preservarla, crea primero un
arreglo flotante antes de asignar.

`np.testing.assert_allclose(actual, expected)` comprueba valores numéricos con
una pequeña tolerancia. `assert_array_equal` exige igualdad de valores y forma.
Si la comprobación pasa no imprime nada; si falla, lanza `AssertionError`.

[Tipos de datos de NumPy](https://numpy.org/doc/stable/user/basics.types.html).

In [7]:
integer_readings = np.array([22, 24, 20], dtype=np.int64)
integer_readings[0] = 22.75
print(integer_readings)

float_readings = np.array([22, 24, 20], dtype=np.float64)
float_readings[0] = 22.75
print(float_readings)
print(np.array([22, "missing"]).dtype)
np.testing.assert_allclose(float_readings[0], 22.75)
np.testing.assert_array_equal(integer_readings, [22, 24, 20])

[22 24 20]
[22.75 24.   20.  ]
<U21


### Ejercicio 2: preservar una temperatura decimal

Crea un arreglo con `22`, `24` y `20` que permita sustituir el primer valor por
`22.75` sin truncarlo. Comprueba el valor con `np.testing.assert_allclose`.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [8]:
decimal_readings = np.array([22, 24, 20], dtype=np.float64)
decimal_readings[0] = 22.75
np.testing.assert_allclose(decimal_readings[0], 22.75)
assert np.issubdtype(decimal_readings.dtype, np.floating)
print(decimal_readings, decimal_readings.dtype)


[22.75 24.   20.  ] float64


## 4. Selección de filas y columnas

En `readings[row, column]`, el primer índice elige la ronda y el segundo la sala.
El slice excluye su límite superior. Un índice entero elimina ese eje del
resultado; un slice lo conserva.

[Selección de elementos y slicing](https://numpy.org/doc/stable/user/basics.indexing.html).

In [9]:
print(readings[0, 1])
print(readings[-1])
print(readings[:3, :2])
print(readings[:, 0])
print(readings[:, 0:1])
print(readings[:, 0].shape, readings[:, 0:1].shape)

24.0
[29. 31. 27.]
[[22. 24.]
 [23. 25.]
 [24. 26.]]
[22. 23. 24. 25. 26. 27. 28. 29.]
[[22.]
 [23.]
 [24.]
 [25.]
 [26.]
 [27.]
 [28.]
 [29.]]
(8,) (8, 1)


Una lista de índices permite escoger columnas no consecutivas. Comprueba el
orden resultante: `[2, 0]` coloca primero el almacén y después la sala norte.

In [10]:
selected_rooms = readings[:, [2, 0]]
print(selected_rooms[:2])
print(selected_rooms.shape)

[[20. 22.]
 [21. 23.]]
(8, 2)


### Ejercicio 3: últimas rondas

Obtén las últimas tres rondas para sala norte y almacén, en ese orden. El
resultado debe ser `[[27, 25], [28, 26], [29, 27]]`, de forma `(3, 2)`.
Comprueba valores y forma.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [11]:
last_three_north_storage = readings[-3:, [0, 2]]
expected_last_three = np.array([[27, 25], [28, 26], [29, 27]])
np.testing.assert_array_equal(last_three_north_storage, expected_last_three)
assert last_three_north_storage.shape == (3, 2)
print(last_three_north_storage)


[[27. 25.]
 [28. 26.]
 [29. 27.]]


### Ejercicio 4: una columna con dos ejes

Selecciona la sala sur conservando la forma `(8, 1)`. Contrasta la expresión
con una selección que devuelva `(8,)`. Explica qué eje desapareció.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [12]:
south_column = readings[:, 1:2]
south_vector = readings[:, 1]
assert south_column.shape == (8, 1)
assert south_vector.shape == (8,)
np.testing.assert_array_equal(south_column[:, 0], south_vector)
print("Column:", south_column.shape, "Vector:", south_vector.shape)
# El índice entero elimina el eje de columnas; el slice 1:2 lo conserva.


Column: (8, 1) Vector: (8,)


## 5. Máscaras booleanas

Una comparación produce un booleano por elemento. Para seleccionar rondas usando
una sala, la máscara debe tener una posición por fila. `&` combina condiciones
por elemento; cada comparación necesita paréntesis.

In [13]:
north_temperatures = readings[:, 0]
warm_rounds = north_temperatures >= 27
print(warm_rounds)
print(warm_rounds.shape)
print(readings[warm_rounds])

comfortable_rounds = (north_temperatures >= 24) & (north_temperatures < 27)
print(readings[comfortable_rounds])

[False False False False False  True  True  True]
(8,)
[[27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]
[[24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]]


Una máscara del mismo tamaño que toda la matriz selecciona celdas y devuelve
un vector. Ya no conserva una fila por ronda. Compara ambas selecciones antes de
usar el resultado en un resumen.

In [14]:
cell_mask = readings >= 28
print(cell_mask.shape)
print(readings[cell_mask])
print(readings[cell_mask].shape)
print(readings[readings[:, 1] >= 28].shape)

(8, 3)
[28. 29. 28. 30. 29. 31.]
(6,)
(4, 3)


### Ejercicio 5: un intervalo

Selecciona las rondas cuya temperatura de la sala sur esté entre `26` incluido
y `29` excluido. Deben quedar tres filas completas. Escribe la máscara, imprime
su forma y comprueba las temperaturas `[26, 27, 28]` de esa columna.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [15]:
south_interval_mask = (readings[:, 1] >= 26) & (readings[:, 1] < 29)
south_interval = readings[south_interval_mask]
assert south_interval_mask.shape == (8,)
assert south_interval.shape == (3, 3)
np.testing.assert_array_equal(south_interval[:, 1], [26, 27, 28])
print(south_interval_mask)
print(south_interval)


[False False  True  True  True False False False]
[[24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]]


## 6. Vistas y copias

Un slice básico produce una vista del mismo búfer. Modificar esa vista puede
cambiar el arreglo de origen. `.copy()` crea datos independientes; la selección
con una lista o un arreglo de índices enteros, o con una máscara booleana,
también obtiene una copia. Un índice entero aislado, como `readings[0]`,
puede devolver una vista de una fila.

Trabajaremos sobre `working_readings` para que el experimento no cambie
`readings`. Observa la diferencia entre alias de datos y archivos: ninguna de
estas asignaciones escribe el CSV.

[Copias y vistas](https://numpy.org/doc/stable/user/basics.copies.html).

In [16]:
working_readings = readings.copy()
north_view = working_readings[:, 0]
north_copy = working_readings[:, 0].copy()
print(np.shares_memory(working_readings, north_view))
print(np.shares_memory(working_readings, north_copy))
north_view[0] = -10
print(working_readings[0, 0], north_copy[0], readings[0, 0])

True
False
-10.0 22.0 22.0


In [17]:
warm_copy = readings[readings[:, 0] >= 27]
warm_copy[0, 0] = -20
print(np.shares_memory(readings, warm_copy))
print(readings[5, 0])

edited_readings = readings.copy()
edited_readings[edited_readings[:, 0] >= 27, 0] = 27
print(edited_readings[:, 0])

False
27.0
[22. 23. 24. 25. 26. 27. 27. 27.]


Obtener una selección con máscara crea una copia, pero asignar directamente
con `array[mask] = value` modifica el arreglo destino. La distinción está en qué
objeto recibe la asignación.

### Ejercicio 6: una selección independiente

Crea `last_rounds` con las dos últimas rondas, cambia su primera temperatura
a `100` y comprueba que `readings` sigue igual que antes. Usa `copy()` y
`np.testing.assert_array_equal` para demostrarlo.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [18]:
original_readings = readings.copy()
last_rounds = readings[-2:].copy()
assert not np.shares_memory(last_rounds, readings)
last_rounds[0, 0] = 100
assert last_rounds[0, 0] == 100
np.testing.assert_array_equal(readings, original_readings)
print(last_rounds)


[[100.  30.  26.]
 [ 29.  31.  27.]]


## 7. Operaciones vectorizadas y resúmenes por eje

En una operación vectorizada expresamos qué cálculo aplicar al arreglo completo.
La conversión de Celsius a Fahrenheit conserva las ocho rondas y las tres salas.
El ciclo siguiente produce el mismo resultado y sirve para comparar las dos formas
de expresar el cálculo; no hace falta medir velocidad con solo 24 valores.

[Operaciones con arreglos](https://numpy.org/doc/stable/user/quickstart.html).


In [19]:
fahrenheit = readings * 9 / 5 + 32
fahrenheit_with_loop = np.array(
    [[temperature * 9 / 5 + 32 for temperature in row] for row in readings]
)
np.testing.assert_allclose(fahrenheit, fahrenheit_with_loop)
print(fahrenheit[:2])
print(fahrenheit.shape)

[[71.6 75.2 68. ]
 [73.4 77.  69.8]]
(8, 3)


### Ejercicio 7: convertir y conservar la forma

Crea `converted_readings` con todas las temperaturas en Fahrenheit mediante una
operación sobre el arreglo, sin escribir un ciclo. Comprueba la forma `(8, 3)`,
que la primera fila sea `[71.6, 75.2, 68.0]` y que `readings` no cambió.


In [20]:
before_conversion = readings.copy()
converted_readings = readings * 9 / 5 + 32
assert converted_readings.shape == (8, 3)
np.testing.assert_allclose(converted_readings[0], [71.6, 75.2, 68.0])
np.testing.assert_array_equal(readings, before_conversion)
print(converted_readings[:2])


[[71.6 75.2 68. ]
 [73.4 77.  69.8]]


### Resumir por eje

Una reducción elimina el eje sobre el que opera. `axis=0` combina las rondas y
conserva las tres salas; `axis=1` combina las salas y conserva las ocho rondas.
Aquí todas las columnas tienen la misma unidad, así que una media por ronda tiene
sentido. Más adelante no promediaremos sin más columnas como edad y tarifa.

[Promedios con `numpy.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html).


In [21]:
room_means = readings.mean(axis=0)
round_means = readings.mean(axis=1)
print(room_means, room_means.shape)
print(round_means[:3], round_means.shape)
np.testing.assert_allclose(room_means, [25.5, 27.5, 23.5])

[25.5 27.5 23.5] (3,)
[22. 23. 24.] (8,)


### Broadcasting: una media distinta para cada sala

`readings` tiene forma `(8, 3)` y `room_means` forma `(3,)`. NumPy alinea las
tres posiciones con las tres columnas y resta cada media a las ocho rondas.
`room_means.reshape(1, 3)` muestra explícitamente esa fila de tres valores;
produce el mismo resultado. NumPy compara las dimensiones desde la derecha: en cada posición deben ser
iguales o una de ellas debe valer 1. Una forma `(8,)` no se alinearía con las
tres columnas. El ejemplo captura ese error para poder continuar.

[Reglas de broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).


In [22]:
deviations = readings - room_means
explicit_row = room_means.reshape(1, 3)
print(readings.shape, room_means.shape, explicit_row.shape)
print(deviations[:2])
assert deviations.shape == readings.shape
np.testing.assert_allclose(deviations, readings - explicit_row)
np.testing.assert_allclose(deviations.mean(axis=0), [0, 0, 0], atol=1e-12)

try:
    readings - round_means
except ValueError as error:
    print("Incompatible shapes:", readings.shape, round_means.shape)
    print(error)

(8, 3) (3,) (1, 3)
[[-3.5 -3.5 -3.5]
 [-2.5 -2.5 -2.5]]
Incompatible shapes: (8, 3) (8,)
operands could not be broadcast together with shapes (8,3) (8,) 


### Ejercicio 9: diferencias respecto de la media

Se resta a cada columna su propia media mediante broadcasting. La forma `(3,)` se alinea con las tres columnas de `(8, 3)` y se repite para cada ronda; una media general perdería la diferencia entre salas.


In [23]:
exercise_room_means = readings.mean(axis=0)
differences = readings - exercise_room_means
assert differences.shape == (8, 3)
np.testing.assert_allclose(differences.mean(axis=0), [0, 0, 0], atol=1e-12)
print("Means:", exercise_room_means)
print("Difference means:", differences.mean(axis=0))


Means: [25.5 27.5 23.5]
Difference means: [0. 0. 0.]


## 8. Mediciones incompletas

`isfinite` identifica valores distintos de `nan` y los infinitos. `all(axis=1)`
exige que las tres posiciones de una ronda sean finitas. Usaremos esa política
para conservar las mismas rondas en todas las salas.

Es una regla del ejercicio: eliminar una fila también descarta mediciones válidas
de sus otras salas. La decisión debe quedar visible en el conteo de rechazos.

[isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html) ·
[all](https://numpy.org/doc/stable/reference/generated/numpy.all.html).

In [24]:
quality_readings = np.loadtxt(
    data_dir / "readings_quality.csv", delimiter=",", skiprows=1, ndmin=2
)
finite_cells = np.isfinite(quality_readings)
complete_rows = finite_cells.all(axis=1)
print(finite_cells)
print(complete_rows)
print(quality_readings[complete_rows])

[[ True  True  True]
 [ True False  True]
 [ True  True  True]
 [ True  True False]]
[ True False  True False]
[[22. 24. 20.]
 [24. 26. 22.]]


### Ejercicio 8: contar los rechazos

Cuenta las filas aceptadas y rechazadas de `quality_readings`. Comprueba que
son dos y dos, y que las medias del arreglo seleccionado son `[23, 25, 21]`.
¿Qué pasaría si no quedara ninguna fila completa?

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [25]:
selected_quality = quality_readings[complete_rows]
accepted_count = int(complete_rows.sum())
rejected_count = int((~complete_rows).sum())
assert accepted_count == 2
assert rejected_count == 2
np.testing.assert_allclose(selected_quality.mean(axis=0), [23, 25, 21])
print("Accepted:", accepted_count, "Rejected:", rejected_count)
print("Means:", selected_quality.mean(axis=0))
# Si no quedara una fila completa, no se calcularía la media de una matriz vacía.


Accepted: 2 Rejected: 2
Means: [23. 25. 21.]


## Continuación: llevar las operaciones al proyecto

En el proyecto `02-reporte-mediciones`, estas operaciones forman funciones que
se importan desde una notebook y desde `main.py`. Sigue la [guía](../GUIA.md) para
cambiar de proyecto y abrir `u2_n2_reporte_mediciones.ipynb`.

Antes de continuar, reinicia el kernel y ejecuta todas las celdas. Los resultados
no deben depender de haber ejecutado una celda fuera de orden.